# Text Representation Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. One-hot by hand.** One slot per vocabulary word, one 1 — and the sparsity maths explains why we need smarter representations.

In [ ]:
import numpy as np

vocab = ["battery", "great", "poor", "price", "screen"]
target = "great"

one_hot = np.zeros(len(vocab))
one_hot[vocab.index(target)] = 1.0
print(one_hot, "<-", target)     # [0. 1. 0. 0. 0.] <- great

V, words_per_doc = 50_000, 150
share = words_per_doc / V
print(f"a {words_per_doc}-word document lights {share:.2%} of a {V:,}-dim vector")
# a 150-word document lights 0.30% of a 50,000-dim vector - almost all zeros!

**2. Your first document-term matrix.** `fit_transform` learns the vocabulary and counts words per document — grammar and order vanish, only counts survive.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

DOCS = [
    "great battery great screen",
    "poor battery poor speaker",
    "great speaker great price",
    "poor screen poor price",
]

bow = CountVectorizer()
X = bow.fit_transform(DOCS)

print("shape:", X.shape)   # (4, 6) -> 4 documents x 6 vocabulary words
pairs = sorted(bow.vocabulary_.items(), key=lambda kv: kv[1])
print("vocabulary_:", pairs)
print("feature names:", list(bow.get_feature_names_out()))
print(X.toarray())         # row i = document i's word counts

**3. Reading vocabulary_ backwards.** `.vocabulary_` maps word → column INDEX, not the reverse — `.get_feature_names_out()` is the inverse lookup.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

DOCS = [
    "great battery great screen",
    "poor battery poor speaker",
    "great speaker great price",
    "poor screen poor price",
]
bow = CountVectorizer().fit(DOCS)

col = bow.vocabulary_["screen"]
print("'screen' lives in column", col)
print("...which is really:", bow.get_feature_names_out()[col])

for word, idx in sorted(bow.vocabulary_.items(), key=lambda kv: kv[1]):
    print(idx, "->", word)

## Part 2 — Practice

**4. Word order? What word order?** Bags erase sequence so opposites collide; bigrams store a slice of word order back.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

pair = ["dog bites man", "man bites dog"]

uni = CountVectorizer(ngram_range=(1, 1)).fit(pair)
same_uni = np.array_equal(uni.transform(pair).toarray()[0],
                          uni.transform(pair).toarray()[1])
print("unigram vectors identical?", same_uni)   # True - BoW cannot tell them apart

bi = CountVectorizer(ngram_range=(1, 2)).fit(pair)
print("with bigrams:", list(bi.get_feature_names_out()))
# ['bites', 'bites dog', 'bites man', 'dog', 'dog bites', 'man', 'man bites']
print("bigram vectors identical?",
      np.array_equal(bi.transform(pair).toarray()[0],
                     bi.transform(pair).toarray()[1]))   # False - order partly restored

**5. Character n-grams rescue typos.** Misspellings share most letter chunks with the correct form — word-level BoW is blind to this.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

words = ["battery", "batery", "bananas"]

char_vec = CountVectorizer(analyzer="char", ngram_range=(2, 4))
sim = cosine_similarity(char_vec.fit_transform(words))

for i in range(len(words)):
    for j in range(i + 1, len(words)):
        print(f"sim({words[i]}, {words[j]}): {sim[i, j]:.2f}")
# The typo 'batery' stays numerically close to 'battery';
# unrelated 'bananas' drifts far away.

**6. IDF by hand.** IDF rewards global rarity: everywhere-words sink toward 1.0, one-document words soar past 1.9.

In [ ]:
import numpy as np

TOY = [
    "battery life is great",
    "battery life is poor",
    "screen is great",
    "price is fair",
]
terms = sorted({t for d in TOY for t in d.split()})
N = len(TOY)

df_counts = np.array([sum(t in d.split() for d in TOY) for t in terms])
idf = np.log((1 + N) / (1 + df_counts)) + 1     # sklearn smooth_idf formula

for t, df_val, idf_val in zip(terms, df_counts, np.round(idf, 3)):
    print(f"{t:>8} | df={df_val} | idf={idf_val}")
# 'is' appears in ALL four documents -> idf 1.0 (no information).
# 'fair'/'poor'/'price'/'screen' appear once -> idf ~1.92 (distinctive!).

**7. Verify TF-IDF against sklearn.** TF-IDF is literally local frequency × global rarity; the default then scales every document vector to length 1.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

TOY = [
    "battery life is great",
    "battery life is poor",
    "screen is great",
    "price is fair",
]
terms = sorted({t for d in TOY for t in d.split()})
N = len(TOY)

df_counts = np.array([sum(t in d.split() for d in TOY) for t in terms])
idf = np.log((1 + N) / (1 + df_counts)) + 1
tf = np.array([[d.count(t) for t in terms] for d in TOY], dtype=float)

hand_tfidf = tf * idf
sk_matrix = TfidfVectorizer(norm=None).fit_transform(TOY).toarray()

print("by-hand matches TfidfVectorizer:", np.allclose(hand_tfidf, sk_matrix))  # True

default_rows = TfidfVectorizer().fit_transform(TOY).toarray()
print("row lengths:", np.round(np.linalg.norm(default_rows, axis=1), 6))
# every row scaled to length 1 -> short tweets compare fairly vs long reviews

## Part 3 — Challenge

**8. A ten-line search engine.** TF-IDF rows come L2-normalized, so cosine ranking reduces to dot products — retrieval in ten lines.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

SEARCH_DOCS = {
    "phone_x": "All-day battery life and fast charging make this phone a travel favourite.",
    "camera_z": "The camera shoots sharp photos at night without any flash.",
    "slim_pro": "Bright screen with slim bezels, though battery endurance is just average.",
    "boom_box": "Loud speakers and surprisingly rich bass for such a tiny size.",
    "gamer_one": "Battery drains quickly during gaming and charging feels slow.",
    "daily_pad": "Crisp display and smooth performance for everyday apps.",
}

engine = TfidfVectorizer()
doc_matrix = engine.fit_transform(list(SEARCH_DOCS.values())).toarray()

def search(query, k=2):
    q = engine.transform([query]).toarray()[0]
    scored = [(name, cosine(q, doc)) for name, doc in zip(SEARCH_DOCS, doc_matrix)]
    return sorted(scored, key=lambda pair: pair[1], reverse=True)[:k]

for query in ["long battery life", "sharp night photos"]:
    print(f"query: '{query}'")
    for name, score in search(query):
        print(f"   {score:.3f}  {name}")
# phone_x owns 'life' and wins the battery query;
# camera_z sweeps 'sharp night photos'.

**9. Spam classifier in a Pipeline.** The Pipeline forces the SAME vectorization at train and predict time — the cheapest insurance against skew.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

MESSAGES = [
    # --- spam ---
    "WINNER! You have been selected for a $1000 gift card. Claim now!",
    "URGENT: your mobile number won a $5000 prize. Call 09099 now",
    "Free entry to win a brand new car! Text WIN to 80085",
    "Congratulations! You are our lucky customer today. Click link to claim",
    "Your account expires tomorrow. Update your details immediately here",
    "Cheap loans approved in minutes. No credit check! Reply YES",
    "Exclusive deal! Buy 1 get 5 free, today only. Shop now!!!",
    "Earn $500 weekly from home. No experience needed. Register today",
    # --- ham ---
    "Hey, running 10 minutes late, order the starters without me",
    "Can you send me the photos from yesterday? They look great",
    "Meeting moved to 3pm tomorrow, conference room B",
    "Thanks for the birthday wishes, had a wonderful day",
    "Did you take your umbrella? It is pouring here",
    "The movie started slow but the ending was brilliant",
    "Left my charger at your place, bring it tonight?",
    "Match cancelled, pitch is waterlogged. Practice Thursday instead",
]
LABELS = ["spam"] * 8 + ["ham"] * 8

X_train, X_test, y_train, y_test = train_test_split(
    MESSAGES, LABELS, test_size=0.25, stratify=LABELS, random_state=42,
)

spam_clf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("naive_bayes", MultinomialNB(alpha=0.5)),
])
spam_clf.fit(X_train, y_train)

predictions = spam_clf.predict(X_test)
print(f"accuracy: {accuracy_score(y_test, predictions):.2f} "
      f"on {len(y_test)} held-out messages\n")
print(classification_report(y_test, predictions, labels=["ham", "spam"]))

for msg in ["You are our lucky WINNER, claim your free prize",
            "See you at practice on Thursday"]:
    print(spam_clf.predict([msg])[0], "<-", msg)
# The Pipeline matters because fit-on-train-only is enforced structurally:
# the test vocabulary can never leak into the training statistics.